In [1]:
from fastapi import FastAPI

print("Hello, World!")

Hello, World!


In [25]:
### interact with llm
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-5-nano")


In [3]:
llm.invoke("Hi")

AIMessage(content='Hi there! How can I help you today? \nTell me what you’re working on or what you’d like to learn, and I’ll dive in.  \nI can explain topics, help with writing or coding, brainstorm ideas, or just chat.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 316, 'prompt_tokens': 7, 'total_tokens': 323, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZZsuI3WMMAjlPe1eqGve3N0gM0ow', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dd39d-3d5b-7a32-ae97-3f36d4846eb4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 316, 'total_tokens': 323, 'input_token_details': {'audio': 0, 'cache_read

In [26]:
llm = ChatOpenAI(
    base_url="https://8000-01kqkwrjxqrt83215rjbqwtrjq.cloudspaces.litng.ai/v1",
    api_key="EMPTY",  # SGLang doesn't require real key
    model="Qwen/Qwen3.6-27B",
    temperature=0.7,
    extra_body={
        "top_k": 20,
        "chat_template_kwargs": {"enable_thinking": False},
    }, 
)

In [58]:
llm.invoke(text[:1012518])

AIMessage(content='Based on the reviews provided, here are the items flagged with a **Safety hazard: 1**:\n\n1.  **Wheatgrass Seeds:** "when I juiced the wheatgrass , the next day I got a horrible rash."\n2.  **CP (Chromium Picolinate) Supplement:** The reviewer describes a dangerous interaction with diabetes medication (metformin), leading to dangerously low blood sugar. They explicitly warn: "Do NOT take if you are already taking an oral hypoglycemic agent!!!"\n3.  **Chips made in China:** The reviewer expresses disgust and concern over food safety laws, stating: "Chinese food safety laws are very different than ours, hence the recalls of tainted foods in the last several years."\n4.  **Amazing Grass Organic Green SuperFood Whole Food Energy Bar:** The reviewer states the product was "PACKED with sugar" and mentions it incited coughing. (Note: While often subjective, the label indicates a safety hazard in this dataset).\n5.  **Fiber Plus Bars (containing Sorbitol):** The reviewer des

In [ ]:
import os
import requests

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {os.environ['RUNPOD_API_KEY']}"
}

data = {
    'input': {"prompt": "Your prompt"}
}

response = requests.post('https://api.runpod.ai/v2/2vr88szp0dmjae/run', headers=headers, json=data)

In [42]:
response.json()

{'status': 403,
 'title': 'Forbidden',
 'detail': 'you do not have permission to perform this action'}

In [48]:

##--max-model-len 262144
## gpu - A100 -80GB * 2
## model size =60GB
## vllm serve --model Qwen/Qwen3.6-27B  ---max-model-len 262144 ---tp=1

# parallism -  data, tensor, pipeline, hybrid
## Data parellism : When your model completely fit into 1 single gpu
##  Tensor parrlism : linear layer of the model split accross mutiple gpu
## pipline parrelism - (1-7) (8-16) ....



## ---max-model-len 262144   ## 262k
## ---max-model-len 8000  ## 8k

## KV cache ---> vllm run --- 2gb ---> 90% percent
## total gpu memory (16GB) = model weigts (2GB) + kV cache (13)GB

## Qwen 0.6. ---32,768 ,  4000

### KV Cache= 2×L×H×D×S×bytes
#L = number of layers, H = no of heads, D=head dimension, S = seq legth , bytes =(8bit) -->BF16 (16 bit. or 2 byytes) =  2bytes

a = 2 * 28 * 8 * 1024 * 4000 * 2

a


3670016000

In [ ]:
### Qwen/Qwen3.6-27B. -  size = 60GB,  max model / context length = 262k
### A100 80GB  - run successffully, --max-model-len 262144 - run
### KV caching: : 20GB
## input -->. hello  -------> 262k
## Total size of KV cache in bytes = (batch_size) * (sequence_length) * 2 * (num_layers) * (hidden_size) *  sizeof(FP16)
## 1 * 262144 * 2 * 64 * 5120 * 2 = 23.8 GB
## 343 GB KV

343597383680

In [51]:
256*4*2*2

4096

In [50]:
# 262. ---> 56 + 17 = 73
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.6-27B")
inputs = tokenizer(text[:1012518], return_tensors="pt")
inputs

{'input_ids': tensor([[   58,   198,   262,  ..., 33739,   314,   295]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]])}

In [51]:
inputs.input_ids.shape

torch.Size([1, 252188])

In [28]:
with open("amazon_food_reviews.json", "r", encoding="cp1252") as f:
    text = f.read()

In [29]:
text[:100]

'[\n    {\n        "Review": "Trident Splash Gum, Strawberry Lime,  9-Piece Packs (Pack of 20): This is'

In [19]:
!wget https://dgoldberg.sdsu.edu/515/amazon_food_reviews.json

--2026-05-02 14:49:38--  https://dgoldberg.sdsu.edu/515/amazon_food_reviews.json
Resolving dgoldberg.sdsu.edu (dgoldberg.sdsu.edu)... 146.244.101.140
Connecting to dgoldberg.sdsu.edu (dgoldberg.sdsu.edu)|146.244.101.140|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2344197 (2.2M) [application/json]
Saving to: ‘amazon_food_reviews.json’

amazon_food_reviews 100%[===================>]   2.24M  1.18MB/s    in 1.9s    

2026-05-02 14:49:41 (1.18 MB/s) - ‘amazon_food_reviews.json’ saved [2344197/2344197]



In [48]:
len(text)/2.3

1012518.2608695653

In [ ]:
text[:1012518]

'[\n    {\n        "Review": "Trident Splash Gum, Strawberry Lime,  9-Piece Packs (Pack of 20): This is the best gum I have ever used and it is sugar free. Wish more stores stocked it today. ",\n        "Safety hazard": "0"\n    },\n    {\n        "Review": "Great deal ! Delicious! ",\n        "Safety hazard": "0"\n    },\n    {\n        "Review": "Barilla Gluten Free Rotini Pasta, 12 Ounce Boxes (Pack of 12): Barilla is my top favorite pasta--especially when I make lasagna!  I cook traditional barilla rotini and also enjoy their wheat pastas!So,  I thought I\'d give this gluten free pasta a try.I was skeptical about how it would taste after I read it was made from corn and rise, but, as always. their pasta tastes great.Like the wheat pasta, you should cook it a little longer.I\'ve only made the wheat pasta and gluten ones with tomato sauce.  I have not yet used it for such recipes as pasta salad or just with butter or olive oil.  Would love to see a review of persons that have done so

In [57]:
252000/22


11454.545454545454

In [ ]:
10080

168.0

In [60]:
!wc amazon_food_reviews.json

   15405  404929 2344197 amazon_food_reviews.json


In [ ]:
### LLM Inference Improvements
## metrics - latencey, througput , itl , ttft

### Batching - continous batching -- vllm default
### kv cache optimization - paged attention , quantization, model parallism, kv cache quantization
### Flash attention --- attnetion calc improve

### KV optimization -  model (arch level change), Multi head, multi query, Grouped Query
## Scaled Dot product attention (SPDA). --->  multi Head attetion ---> Multi Query attention

## Attention = K,V,Q. -- 32 heads

### 32k , 32v, 32Q

## multi query. --->. 32Q ---> 1 shared k ,v

##  Grouped Query. ->. 32Q, 8k,8v (balanced accury, reduced size)


### Paged Attention
### To reduce memory fragemnetion
#### [.............] kv cache token : 12
## req A [6]--> [AAAAAA]
## req B [4]---->[AAAAAA_BBBB_]
## req C [2] ---> [AAAAAA_BBBB_]]

## paged attetion res A, B = [A1, A2, B7, B]

## kv cache =  12 ---> memory fragemnetion


### Flash Attention --> Inference
### GPU (model weihts - VRAM)
### GPU (processor(SRAM + memory(VRAM)))
### GPU Faster (Compute excellent , memoey bandwidtgh bound)
###  calc in chunks ---> SRAM





In [41]:
from openai import OpenAI

client = OpenAI(
    api_key="",
    base_url="https://pwyob056uc30ab-8000.proxy.runpod.net/v1"
)

response = client.chat.completions.create(
    model="Qwen/Qwen3.5-0.8B",
    messages=[
        {"role": "user", "content": "Type \"I love Qwen3.5\" backwards"}
    ],
    max_tokens=81920,
    temperature=1.0,
    top_p=0.95,
    presence_penalty=1.5,
    extra_body={
        "top_k": 20,
        "enable_thinking": True,
    }, 
    )

print(response.choices[0].message.content)

I love Qwen3.5


In [53]:
from openai import OpenAI
# Configured by environment variables

messages = [
    {"role": "user", "content": "Hi"},
]

chat_response = client.chat.completions.create(
    model="Qwen/Qwen3.5-0.8B",
    messages=messages,
    max_tokens=81920,
    temperature=1.0,
    top_p=0.95,
    presence_penalty=1.5,
    extra_body={
    "chat_template_kwargs": {
        "enable_thinking": True
    }
}
)
print("Chat response:", chat_response)


Chat response: ChatCompletion(id='chatcmpl-83c604e4e32e8e8b', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Thinking Process:\n\n1.  **Analyze the Input:**\n    *   User input: "Hi"\n    *   Intent: Greeting. Polite message to acknowledge the user.\n    *   Tone: Friendly, casual (but not overly formal).\n\n2.  **Determine the appropriate response:**\n    *   Acknowledge the greeting with a friendly response.\n    *   Keep it concise but warm.\n    *   Optionally offer help (if appropriate) but first focus on gifting.\n\n3.  **Drafting the response:**\n    *   Option 1: Hello! I\'m ready to help. What can I do for you?\n    *   Option 2: Hi there! How can I assist you today?\n    *   Option 3: Greetings! Please feel free to chat. I\'d love to answer any questions.\n\n4.  **Selecting the best option:** Option 1 or 2 is safer and more direct. Let\'s go with a friendly opener offering help.\n\n5.  **Refining:**\n    *   "Hello!" -> "G

In [54]:
chat_response.choices[0].message.content

'Thinking Process:\n\n1.  **Analyze the Input:**\n    *   User input: "Hi"\n    *   Intent: Greeting. Polite message to acknowledge the user.\n    *   Tone: Friendly, casual (but not overly formal).\n\n2.  **Determine the appropriate response:**\n    *   Acknowledge the greeting with a friendly response.\n    *   Keep it concise but warm.\n    *   Optionally offer help (if appropriate) but first focus on gifting.\n\n3.  **Drafting the response:**\n    *   Option 1: Hello! I\'m ready to help. What can I do for you?\n    *   Option 2: Hi there! How can I assist you today?\n    *   Option 3: Greetings! Please feel free to chat. I\'d love to answer any questions.\n\n4.  **Selecting the best option:** Option 1 or 2 is safer and more direct. Let\'s go with a friendly opener offering help.\n\n5.  **Refining:**\n    *   "Hello!" -> "Greetings!"\n    *   "I am ready..." -> "Here is to interacting."\n    *   Wait, keep it simple. "Hi there!" is fine. But since I\'m an AI, acknowledging the state

Initializing a V1 LLM engine (v0.21.0) with config: model='Qwen/Qwen3.5-0.8B', speculative_config=None, tokenizer='Qwen/Qwen3.5-0.8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=262144, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collect_detailed_traces=None, kv_cache_metrics=False, kv_cache_metrics_sample=0.01, cudagraph_metrics=False, enable_layerwise_nvtx_tracing=False, enable_mfu_metrics=False, enable_mm_processor_stats=False, enable_logging_iteration_details=False), seed=0, served_model_name=Qwen/Qwen3.5-0.8B, enable_prefix_caching=False, enable_chunked_prefill=True, pooler_config=None, compilation_config={'mode': <CompilationMode.VLLM_COMPILE: 3>, 'debug_dump_path': None, 'cache_dir': '', 'compile_cache_save_format': 'binary', 'backend': 'inductor', 'custom_ops': ['none'], 'ir_enable_torch_wrap': True, 'splitting_ops': ['vllm::unified_attention_with_output', 'vllm::unified_mla_attention_with_output', 'vllm::mamba_mixer2', 'vllm::mamba_mixer', 'vllm::short_conv', 'vllm::linear_attention', 'vllm::plamo2_mamba_mixer', 'vllm::gdn_attention_core', 'vllm::gdn_attention_core_xpu', 'vllm::olmo_hybrid_gdn_full_forward', 'vllm::kda_attention', 'vllm::sparse_attn_indexer', 'vllm::rocm_aiter_sparse_attn_indexer', 'vllm::deepseek_v4_attention', 'vllm::unified_kv_cache_update', 'vllm::unified_mla_kv_cache_update'], 'compile_mm_encoder': False, 'cudagraph_mm_encoder': False, 'encoder_cudagraph_token_budgets': [], 'encoder_cudagraph_max_vision_items_per_batch': 0, 'encoder_cudagraph_max_frames_per_batch': None, 'compile_sizes': [], 'compile_ranges_endpoints': [2048], 'inductor_compile_config': {'enable_auto_functionalized_v2': False, 'size_asserts': False, 'alignment_asserts': False, 'scalar_asserts': False, 'combo_kernels': True, 'benchmark_combo_kernel': True}, 'inductor_passes': {}, 'cudagraph_mode': <CUDAGraphMode.FULL_AND_PIECEWISE: (2, 1)>, 'cudagraph_num_of_warmups': 1, 'cudagraph_capture_sizes': [1, 2, 4, 8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 96, 104, 112, 120, 128, 136, 144, 152, 160, 168, 176, 184, 192, 200, 208, 216, 224, 232, 240, 248, 256, 272, 288, 304, 320, 336, 352, 368, 384, 400, 416, 432, 448, 464, 480, 496, 512], 'cudagraph_copy_inputs': False, 'cudagraph_specialize_lora': True, 'use_inductor_graph_partition': False, 'pass_config': {'fuse_norm_quant': False, 'fuse_act_quant': False, 'fuse_attn_quant': False, 'enable_sp': False, 'fuse_gemm_comms': False, 'fuse_allreduce_rms': False, 'fuse_act_padding': False}, 'max_cudagraph_capture_size': 512, 'dynamic_shapes_config': {'type': <DynamicShapesType.BACKED: 'backed'>, 'evaluate_guards': False, 'assume_32_bit_indexing': False}, 'local_cache_dir': None, 'fast_moe_cold_start': False, 'static_all_moe_layers': []}, kernel_config=KernelConfig(ir_op_priority=IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native']), enable_flashinfer_autotune=False, moe_backend='auto')


### max_seq_length = 262k
GPU KV cache size: 1,460,979 tokens
Maximum concurrency for 262,144 tokens per request: 5.57x

## max = 26k